# Submit a delivery with the Python client 

Build a delivery step by step, validate it with a dry run, submit it and follow its status.

You need a Pushing Entity account. See the [credentials page](../credentials.md). If not set, the following cell will prompt for them and store them in the environment.


In [ ]:
import getpass
import os

# Only asks for what is not already in the environment. getpass does not echo or save what you type.
for name in ["COPERNICUSMARINE_SERVICE_USERNAME", "COPERNICUSMARINE_SERVICE_PASSWORD"]:
    if not os.environ.get(name):
        os.environ[name] = getpass.getpass(f"{name}: ")

## 1. Build the delivery

A delivery is a list of operations that run in order. Operations take keyword arguments, and you can keep adding to the delivery before submitting it.

Use a real product and dataset of your Pushing Entity.

In [ ]:
PRODUCT_ID = "<MY_PRODUCT>"
DATASET_ID = "<MY_DATASET_WITH_TAG>"

In [ ]:
from pathlib import Path

from copernicusmarine_delivery import Delete, Delivery, Upload, delivery_status

# Local files to upload
Path("test_data/local/files").mkdir(parents=True, exist_ok=True)
Path("test_data/local/files/file_01.nc").write_text("hello 1")
Path("test_data/local/files/file_02.nc").write_text("hello 2")

delivery = Delivery(operations=[Delete(files=["test.nc"])])
delivery.add(Upload(files=["my/local/files/file_01.nc", "my/local/files/file_02.nc"]))

for operation in delivery.operations:
    print(operation)

## 2. Validate with a dry run

With `dry_run=True`, the product, the dataset and the files are validated, but nothing is uploaded and no delivery is submitted.

In [ ]:
response = delivery.submit(product_id=PRODUCT_ID, dataset_id=DATASET_ID, dry_run=True)

print("Fatal error:", response.fatal_error)

## 3. Explore the response

The response has one entry per operation, in the same order as the delivery.

In [ ]:
for operation_response in response.operations_responses:
    print(type(operation_response).__name__)
    print("  files to delete:", getattr(operation_response, "files_to_delete", None))
    print("  files uploaded:", getattr(operation_response, "files_uploaded", None))
    print("  files failed:", getattr(operation_response, "files_failed", None))

## 4. Submit for real

Same call without `dry_run`. This uploads the files and submits the delivery.

In [ ]:
response = delivery.submit(product_id=PRODUCT_ID, dataset_id=DATASET_ID)

if response.fatal_error:
    print(f"Delivery failed: {response.fatal_error}")
else:
    print(f"Delivery ID: {response.delivery_id}")

## 5. Follow the status

The delivery is processed after it is submitted. Check its status until it is `completed` or `failed`.

In [ ]:
import time

assert response.delivery_id
for _ in range(10):
    status = delivery_status(response.delivery_id)
    print(status.status.value)
    if status.status.value in ("completed", "failed"):
        break
    time.sleep(10)

print(status.error_message)